# panoloc

Starter notebook.

In [2]:
!uv add requests pandas

Resolved 96 packages in 457ms                                        
Prepared 2 packages in 833ms                                             
Installed 2 packages in 17ms                                
 + numpy==2.5.3
 + pandas==3.0.6


In [3]:
import sys
import requests
import math
import pandas as pd
from pathlib import Path

print(sys.version)
print(Path.cwd())

3.12.12 (main, Feb 12 2026, 00:42:14) [Clang 21.1.4 ]
/home/jd/Desktop/panoloc


## Get Panorama Around X meters

In [4]:
PIC_ID = "3b419b72-b710-4508-b96b-a1925ba44d34"
RADIUS_M = 10
SAME_SEQUENCE = True
LIMIT_SEQ = 1000

In [19]:
API = "https://api.panoramax.xyz/api"

# 1 - Get the target picture
r = requests.get(f"{API}/search", params={"ids": PIC_ID})
r.raise_for_status()

# print(r.json()["features"])
# print(r.json()["collection"])

if not r.json()["features"]:
    raise("No Features Error")

target = r.json()["features"][0]
print(f"Running the algorithm on {target["links"][0]}")

lon, lat = target["geometry"]["coordinates"]
target_seq = target["collection"]
print(f"Target Seq Id : {target_seq}")
print(f"lon , lat {lon} {lat}")


# 2 - Get sequence of images

# find the bbox
dlat = RADIUS_M / 111_320
dlon = dlat / math.cos(math.radians(lat))
bbox = f"{lon - dlon},{lat - dlat},{lon + dlon},{lat + dlat}"

items = []
url = f"{API}/collections/{target_seq}/items"
params = {"bbox": bbox, "limit": LIMIT_SEQ}
while url:
    data = requests.get(url, params=params).json()
    items += data["features"]
    url = next((l["href"] for l in data.get("links", []) if l.get("rel") == "next"), None)
    params = None  #next already contains the parameters

# 3 - Build the table from the bbox results
df = pd.DataFrame([{
    "id": f["id"],
    "sequence": f["collection"],
    "datetime": f["properties"].get("datetime"),
    "lon": f["geometry"]["coordinates"][0],
    "lat": f["geometry"]["coordinates"][1],
    "thumb": f"{API}/pictures/{f['id']}/thumb.jpg",
    "hd": f"{API}/pictures/{f['id']}/hd.jpg",
} for f in items])

# keep only the target's sequence
df = df[df["sequence"] == target_seq]

# # 4 - Order capture by distance
# dx = (df["lon"] - lon) * 111_320 * math.cos(math.radians(lat))
# dy = (df["lat"] - lat) * 111_320
# df["distance_m"] = ((dx**2 + dy**2) ** 0.5).round(1)

# 4- Order capture by time
df = df.sort_values("datetime").reset_index(drop=True)

# df = df.sort_values("distance_m").reset_index(drop=True)
print(f"{len(df) - 1} other pictures found in the bbox")
pd.set_option("display.max_colwidth", None)
display(df["thumb"])



Running the algorithm on {'rel': 'license', 'href': 'https://spdx.org/licenses/CC-BY-SA-4.0.html', 'title': 'License for this object (CC-BY-SA-4.0)'}
Target Seq Id : 2f25fcd0-167e-48cf-9806-8a78b9120ec1
lon , lat 4.3674329 50.8427082
129 other pictures found in the bbox


0      https://api.panoramax.xyz/api/pictures/f65ade8f-b8c7-4a97-bec9-d92e1a58ed9b/thumb.jpg
1      https://api.panoramax.xyz/api/pictures/82a9fbc5-0201-4d3c-8294-d7b38012b35f/thumb.jpg
2      https://api.panoramax.xyz/api/pictures/a965fb21-d0b5-4942-a431-5cfe5776897a/thumb.jpg
3      https://api.panoramax.xyz/api/pictures/8b66c642-5736-45cc-8c05-8af132927f46/thumb.jpg
4      https://api.panoramax.xyz/api/pictures/745f5dfb-d929-4a50-9289-046ffc2aff64/thumb.jpg
                                               ...                                          
125    https://api.panoramax.xyz/api/pictures/ae0b88ef-7924-429e-8468-2773c6c78d2c/thumb.jpg
126    https://api.panoramax.xyz/api/pictures/90f41b38-aeae-4b7c-a24a-85754d08f9d5/thumb.jpg
127    https://api.panoramax.xyz/api/pictures/5faf0c56-42af-4336-992b-26f5278ab2cd/thumb.jpg
128    https://api.panoramax.xyz/api/pictures/4e5249ec-f868-468a-844b-b4e0eec3eb0a/thumb.jpg
129    https://api.panoramax.xyz/api/pictures/8934c82e-68e8-4891-8537-